## Using edgeR for DEG analysis between clusters -- comparing region_leiden clusters within capillary cells and using pseurod-bulk approach

In [ ]:
### identify cluster marker genes
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    library(harmony)
    library(reticulate)
    library(viridis)
    library(RColorBrewer)
    library(ComplexHeatmap)
    library(colorRamp2)
    library(edgeR)
})

# set large memory  
options(future.globals.maxSize = 120*1024^3)

## plotting parameters
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))

## set working dir
dir = "/home/kibr/Work/Vascular_atlas"
setwd(dir)

In [ ]:
## Reloading 
# Convert h5ad to h5seurat
h5ad_file="./Results/Revision_3/Endo_cleaned.h5ad"
obj_oi = schard::h5ad2seurat(h5ad_file)
print(obj_oi)

In [ ]:
### optional: check how many brain regions are there
length(unique(obj_oi$brain_region))
table(obj_oi$brain_region,obj_oi$region_leiden)
table(obj_oi$individualID)

In [ ]:
### assign back to Seurat object
meta = obj_oi@meta.data
meta$region_cluster = gsub('Cluster_','C', meta$region_cluster)
obj_oi@meta.data = meta
table(obj_oi$region_cluster,useNA = "always")

## check cell type
table(meta$Cell_type)

<!-- ### ---------------------------------------------------------------------
### Calculating module score for BBB signature
### --------------------------------------------------------------------- -->

In [ ]:
# ## Extra libraries for gene set scoring
# library(ggpubr)

# ## Nrmalize the data for gene set scoring
# obj_oi <- NormalizeData(obj_oi)

# ## BBB signature genes
# ## read in the GO terms related to transport BBB function
# GO_term <- read.delim("./Data/GO_terms/transport across blood-brain barrier.gaf",comment.char = "!",header=F)
# genes <- unique(GO_term$V3)
# genes <- genes[genes %in% rownames(obj_oi[['RNA']])]
# length(genes)

# ### Calculate the module scores
# obj_oi <- AddModuleScore(
#   obj_oi,
#   list(genes),
#   pool = NULL,
#   nbin = 20,
#   ctrl = 100,
#   k = FALSE,
#   assay = "RNA",
#   name = "Transport_across_BBB",
#   seed = 1,
#   search = FALSE,
#   slot = "data",
# )

In [ ]:
# col1=c(
#   "Large_Artery"     = "#B2182B",  # deep red
#   "Arterial"        = "#F4A582",  # light orange
#   "Capillary"       = "#66C2A5",  # medium teal
#   "Venous"           = "#4393C3",  # medium blue
#   "Fenestrated_Capillary"   = "#FDAE61",  # golden orange (distinct capillary subtype)
#   "EndoMT"          = "#BC80BD"  # soft lavender purple
# )

In [ ]:
# ## plotting
# p1 <- FeaturePlot(obj_oi,features="Transport_across_BBB1",reduction = "umapharmony_",pt.size = 3,label = TRUE)+umap_theme+
#             scale_colour_gradientn(colours = rev(brewer.pal(n = 11, name = "RdBu")[1:9]))
# p2 <- DimPlot(obj_oi,group.by = c('Cell_type'),reduction = "umapharmony_",pt.size = 3,cols = col1,label = TRUE,label.size = 10)+umap_theme+NoLegend()
# p3 <- DimPlot(obj_oi,group.by = c('region_cluster'),reduction = "umapharmony_",pt.size = 3)+umap_theme

# options(repr.plot.width = 16,repr.plot.height = 12)
# p2
# p1
# p3

# # ggsave(filename = "./Results/Revision_2/Figures/Figure_1X_Endothelial_cell_type_umap.pdf",
# #     patchwork::wrap_plots(p1,p2,p3,ncol = 1),
# #       scale = 1, width = 14, height = 25)


### Box plot to show the distribution of the BBB transport score across different clusters

In [ ]:
# ## colors code
# col1=c(
#   "C5"     = "#B2182B",  # deep red
#   "Artery"           = "#D6604D",  # red-orange
#   "C3sds"        = "#F4A582",  # light orange
#   "C1"       = "#A6DBD0",  # light teal
#   "C2"       = "#66C2A5",  # medium teal
#   "C3"       = "#41AE76",  # deeper teal
#   "Venule"           = "#4393C3",  # medium blue
#   "C4"             = "#2166AC",  # dark blue
#   "Fenestrated_EC"   = "#FDAE61",  # golden orange (distinct capillary subtype)
#   "EndoMT1"          = "#BC80BD",  # soft lavender purple
#   "C4wew"          = "#762A83"   # deep plum purple
# )

In [ ]:
# ## check which region cluster contain higher proportion of BBB signature
# meta = obj_oi@meta.data

# my_comparisons = list(c("C1","C2"),c("C1","C3"),c("C2","C3"))

# p<-ggplot(meta, aes(x=region_cluster, y=Transport_across_BBB1, fill=region_cluster)) +scale_fill_manual(values = col1)+
#   geom_boxplot() + theme_classic()+ theme(axis.text.x = element_text(angle = 45, hjust = 1))+
#   stat_compare_means(comparisons = my_comparisons,na.rm = T,method = "t.test",
#         symnum.args =list(cutpoints = c(0, 0.0001, 0.001, 0.01, 0.05, Inf), symbols = c("****", "***", "**", "*", "ns")))
# p
# ## save plot for the BBB signature score across region clusters
# # ggsave(p,filename = "./Results/Revision_2/Figures/Figure_1X_Endothelial_BBB_signature_boxplot.pdf",width = 4.5,height = 5)

### ---------------------------------------------------------------------
### Cell type DEG 
### ---------------------------------------------------------------------

In [ ]:
# Idents(obj_oi) = "Cell_type"
# obj_oi <- NormalizeData(obj_oi)

# table(Idents(obj_oi))

In [ ]:
# #### code for identifying the cell type makers ####
# # the parameters are based on https://www.cell.com/cell/fulltext/S0092-8674(23)00973-X
# table(Idents(obj_oi))
# # clean <- PrepSCTFindMarkers(clean)
# # running differential analysis
# all_markers_major <- FindAllMarkers(object = obj_oi,verbose = T,min.pct = 0.25,logfc.threshold = 0.25
# #,test.use = "MAST" #might use MAST model
# ) 
# write.csv(all_markers_major,file = "./Results/Revision_2/DEG/Table_S3_Endothelial_cell_type_markers.csv")

# table(all_markers_major$cluster)

In [ ]:
# meta = obj_oi@meta.data
# table(meta$region_name,meta$Cell_type)

## Get psudo-bulk expression

In [ ]:
## getting the pseudo-bulk object
pseudo_obj <- AggregateExpression(obj_oi, assays = "RNA", return.seurat = T, group.by = c("sampleID","region_cluster"))
pseudo_obj

## getting the count matrix for edgeR analysis
mat = pseudo_obj[['RNA']]$counts
head(mat)

## getting the meta data for edgeR analysis
meta = pseudo_obj@meta.data
group = meta$region_cluster
head(meta)

In [ ]:
## Build edgeR format
y <- DGEList(counts=mat,group=group)
cpm_mat <- cpm(y, log=F)

## add donor info
df = str_split_fixed(rownames(y$samples),"-",4)[,c(1,2)]
y$samples$donor = paste(df[,1],df[,2],sep = "_")

## filter sample by number of reads
summary(y$samples$lib.size)

## Removing the samples with low number of reads lower than 5% quantile
cutoff <- quantile(y$samples$lib.size, 0.05)
print(cutoff)
keep.samples <- y$samples$lib.size > cutoff 
table(keep.samples)
y <- y[, keep.samples]
cpm_mat = cpm_mat[, keep.samples]

## preprocessing for model fitting
group <- droplevels(factor(y$samples$group))
donor <- droplevels(factor(y$samples$donor))

design <- model.matrix(~0 + group + donor)
colnames(design) <- make.names(colnames(design))
head(design)

## filter genes based on design
keep <- filterByExpr(y, design=design)
table(keep)
y <- y[keep, , keep.lib.sizes=FALSE]

In [ ]:
## normalize library sizes
y <- normLibSizes(y)
summary(y$samples$norm.factors)

options(repr.plot.width=12,repr.plot.height=12)
brain_region <- as.factor(y$samples$group)
plotMDS(y, pch=16, col=c(1:36)[y$samples$group], main="brain region",labels = y$samples$donor)

In [ ]:
## estimate dispersion
y <- estimateDisp(y, design, robust=TRUE)
##--------- fit the model -----------##
fit <- glmQLFit(y, design, robust=TRUE)

In [ ]:
## define contrasts for each brain region vs all others
K <- nlevels(group)
grp_cols <- grep("^group", colnames(design), value=TRUE)

# sanity: length(grp_cols) should be K
stopifnot(length(grp_cols) == K)

C <- matrix(-1/(K-1), nrow=K, ncol=K,
            dimnames=list(grp_cols, levels(group)))
diag(C) <- 1

# expand to full design rows (add zeros for donor columns)
Cfull <- matrix(0, nrow=ncol(design), ncol=K,
                dimnames=list(colnames(design), colnames(C)))
Cfull[grp_cols, ] <- C

qlf <- vector("list", K)
for(i in seq_len(K)) {
  qlf[[i]] <- glmQLFTest(fit, contrast=Cfull[, i])
  qlf[[i]]$comparison <- paste0(levels(group)[i], "_vs_others")
}


In [ ]:
res = data.frame()
for (i in 1:length(qlf)){
    df = qlf[[i]]$table
    df$genes = rownames(df)
    df$FDR = p.adjust(df$PValue, method = "bonferroni")
    df %>%
        dplyr::arrange(FDR) %>%
        # dplyr::filter(FDR  < 0.05) %>%
        # dplyr::filter(logFC > 0) %>% 
        ungroup() -> sig

    if (nrow(sig) == 0) next    
    
    sig$cluster = levels(group)[i]
    res = rbind(res,sig)
}

In [ ]:
res$direction = ifelse(res$logFC > 0, "up", "down")
table(res$cluster,res$direction)

In [ ]:
## write the significant results
write.csv(res,file = "./Results/Revision_3/DEG/Endo_rg_clusters_edgeR_wC5.csv")

In [ ]:
dt <- lapply(lapply(qlf, decideTests), summary)
dt.all <- do.call("cbind", dt)
dt.all

In [ ]:
res_sig = res %>% dplyr::filter(abs(logFC) > 0.25) %>% dplyr::filter(FDR < 0.05)
table(res_sig$cluster,res_sig$direction)

In [ ]:
write.csv(res_sig,file = "./Results/Revision_3/DEG/Endo_rg_clusters_edgeR_wC5_sig.csv")

## Draw Heatmap of top DE genes

In [ ]:
## optional: reloading the res
res = read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_edgeR_wC5.csv", row.names = 1)
table(res$cluster,res$direction)

In [ ]:
## Significant positive DEGs, excluding MYH11
sig <- res %>%
  filter(logFC > 0.5, FDR < 0.05, !genes %in% c('MYH11','COL5A2'))

## 1. Top 5 DEGs for each cluster
top5 <- sig %>%
  group_by(cluster) %>%
  arrange(FDR) %>%
  slice_head(n = 5) %>%
  ungroup()

## 2. Genes that are significant in only ONE cluster
unique_deg <- sig %>%
  group_by(genes) %>%
  filter(n_distinct(cluster) == 1) %>%
  ungroup()

## Top 5 cluster-specific DEGs for each cluster
top5_unique <- unique_deg %>%
  group_by(cluster) %>%
  arrange(FDR) %>%
  slice_head(n = 5) %>%
  ungroup()

## genes that you manually want to include
genes_to_add <- c(
  "SLC7A1",
  "SLC39A8",
  "SLC38A6"
)

## extract their significant positive DEG row
extra_genes <- sig %>%
  filter(genes %in% genes_to_add) %>%
  group_by(genes) %>%
  slice_max(logFC, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  mutate(selection = "Top5_unique")

top <- bind_rows(
  top5 %>% mutate(selection = "Top5"),
  top5_unique %>% mutate(selection = "Top5_unique"),
  extra_genes
) %>%
  distinct(cluster, genes, .keep_all = TRUE)

top <- top %>%
  mutate(
    selection = factor(
      selection,
      levels = c("Top5", "Top5_unique")
    )
  ) %>%
  arrange(cluster,selection,desc(logFC))

In [ ]:
excluded_genes <- c(
  "NXN",
  "PPFIBP1",
  "AGO4",
  "ADAMTS10"
)
top = top %>%
  filter(!genes %in% excluded_genes)

In [ ]:
## working on the top regulated first
df_all = res

## construct the matrix
mat<-matrix(nrow=length(unique(top$genes)), ncol=length(unique(df_all$cluster)))
colnames(mat) <- unique(df_all$cluster)

### create the matrix to show the gene expression change pattern
for (j in 1:length(unique(top$genes))){
    for (i in 1:length(unique(df_all$cluster))){
        geneTemp<-unique(top$genes)[j]
        rgTemp<-colnames(mat)[i]

        sub<-df_all[which(df_all$genes==geneTemp & df_all$cluster==rgTemp),]
        if (nrow(sub)>0){
        mat[j,i]<-sub$logFC
        }
        else{
        mat[j,i]<-0
        }
    }  
}
rownames(mat)<- unique(top$genes)


## generating a significance matrix:
## TRUE only when FDR < 0.05 AND logFC > 0.5
sig_mat <- matrix(FALSE,nrow = length(unique(top$genes)),ncol = length(unique(df_all$cluster)))
colnames(sig_mat) <- unique(df_all$cluster)

for (j in 1:length(unique(top$genes))) {
  for (i in 1:length(unique(df_all$cluster))) {

    geneTemp <- unique(top$genes)[j]
    rgTemp <- colnames(sig_mat)[i]

    sub <- df_all[
      df_all$genes == geneTemp &
      df_all$cluster == rgTemp,
    ]

    if (nrow(sub) > 0) {
      sig_mat[j, i] <- sub$FDR < 0.05 & abs(sub$logFC)  > 0.5
    }
  }
}

rownames(sig_mat) <- unique(top$genes)

## reorder the matrix
mat <- mat[,order(as.numeric(str_split_fixed(colnames(mat),pattern = "_",2)[,1]))]
sig_mat <- sig_mat[,order(as.numeric(str_split_fixed(colnames(sig_mat),pattern = "_",2)[,1]))]
# mat
# sig_mat

In [ ]:
pdf("./Results/Revision_3/DEG/Figure_2X_Endothelial_DEG_heatmap_wC5.pdf",width = 4,height = 12)
ht=Heatmap(mat, cluster_rows=F,cluster_columns=F,
           col=colorRamp2(c(-2,0, 2),c("#4575B4", "#F7F7F7", "#D73027")),  
#           top_annotation=hl, #name="Number of genes with significant expression changes",  
          #  right_annotation = hl,
           show_column_names=T, show_row_names=T, column_title=NULL,
           row_names_side="left", row_title_gp=gpar(fontsize=60),
           row_names_max_width = max_text_width(rownames(mat), gp = gpar(fontsize = 24)),
           cell_fun = function(j, i, x, y, w, h, fill) {
  if(sig_mat[i, j]) {grid.text("*", x, y, gp=gpar(fontsize=35, col="black"), vjust="center")}
  #if(sig_mat[i, j] <0.05 & sig_mat[i, j] >0.01){grid.text("*", x, y, gp=gpar(fontsize=35, col="black"), vjust="center")}
})

ht
dev.off()

### Check specific genes among individuals

In [ ]:
unique(res$cluster)
head(res[res$cluster == "C1" & res$direction == "up",],20)

In [ ]:
### Function for box plot
plot_gene_boxplot <- function(gene_name, cpm_mat, y, custom_colors = NULL) {
  
  # Default colors if none provided
  if (is.null(custom_colors)) {
    custom_colors <- c(
      "Cluster-1" = "#4a9dbf",
      "Cluster-2" = "#c9924e",
      "Cluster-3" = "#4a7a62",
      "Cluster-4" = "#c4625a",
      "Cluster-5" = "#8a7aaa"
    )
  }
  
  # Validate gene exists in matrix
  if (!gene_name %in% rownames(cpm_mat)) {
    stop(paste("Gene", gene_name, "not found in cpm_mat"))
  }
  
  # Prepare data
  target_df <- as.data.frame(cpm_mat[gene_name, ])
  colnames(target_df) <- "Expression"
  target_df$Region_Cluster <- factor(y$samples$group)
  
  # Create plot
  p <- ggplot(target_df, aes(x = Region_Cluster, y = Expression)) +
    geom_boxplot(aes(fill = Region_Cluster)) +
    geom_point(position = position_jitter(width = 0.2), alpha = 0.3) +
    scale_fill_manual(values = custom_colors) +
    labs(
      y = "Normalized Expression (CPM)",
      x = "Region Cluster"
    ) +
    theme_classic() +
    theme(axis.text.x = element_text(angle = 330, hjust = 0, vjust = 1))
  
  # Italic gene name title
  p <- p + ggtitle(bquote(italic(.(gene_name))))
  
  return(p)
}

In [ ]:
p1 = plot_gene_boxplot("THSD4", cpm_mat, y)
p2 = plot_gene_boxplot("SLC7A1", cpm_mat, y)
p3 = plot_gene_boxplot("SLC38A6", cpm_mat, y)
p4 = plot_gene_boxplot("PLA1A", cpm_mat, y)
p5 = plot_gene_boxplot("MGP", cpm_mat, y)
p1
# ggsave(filename = "./Results/Revision_2/DEG/Figure_2h_Endothelial_region_cluster_DEG_examples.pdf", patchwork::wrap_plots(p1,p2,p3,p4,p5, ncol = 1),scale = 1, width = 5, height = 18)

## Draw upset plot for the significant genes across clusters

In [ ]:
## reloading the DEG results for the heatmap plot
res = read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_edgeR_wC5.csv",row.names = 1)
head(res)

In [ ]:
## get the significant genes
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  group_by(cluster) 

## check number of significant genes in each cluster
table(sig_genes_list$cluster)

In [ ]:
## Draw upset plot for the significant genes across clusters
library(UpSetR)
options(repr.plot.height = 6,repr.plot.width = 12)

## prepare the data for upset plot
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  # dplyr::filter(logFC < -0.5) %>%
  group_by(cluster) 
sig_genes_list = split(sig_genes_list$genes, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
# length(sig_genes_list)

## draw the upset plot and add title "Number of upregulated DEGs in each region cluster"
pdf(file = "./Results/Revision_3/DEG/Endothelial_rg_clusters_sig_genes_upset_plot_WC5.pdf",width = 12,height = 6)
upset(
  fromList(sig_genes_list), 
  order.by = "freq", 
  nsets = length(sig_genes_list), 
  nintersects =NA,
  main.bar.color = c("#FF0000","#0f0f0f","#03B8D8","#0f0f0f","#009E73", "#CC79A7","#009E73","#009E73","#0f0f0f","#0f0f0f", "#0f0f0f","#0f0f0f","#0f0f0f","#0f0f0f"),
  sets.bar.color = c("#FF0000","#03B8D8", "#009E73", "#009E73", "#CC79A7"),
  text.scale = c(2, 2, 2, 2, 3, 2),
  point.size = 5, line.size = 1.6, mb.ratio = c(0.6, 0.4),
  att.pos = "bottom",
  mainbar.y.label = "Number of upregulated DEGs",
  sets.x.label = "Region Clusters"
  )
dev.off()

### Enrichment analysis for the sig genes between clusters and others


In [ ]:
res = read.csv("./Results/Revision_2/DEG/Endo_rg_clusters_edgeR_wC5.csv",row.names = 1)
res_sig = res %>% dplyr::filter(FDR < 0.05) %>% dplyr::filter(logFC > 0.5)
table(res_sig$cluster)
# res

In [ ]:
suppressPackageStartupMessages({
library(clusterProfiler)
library('org.Hs.eg.db')
# library(ReactomePA)
})

In [ ]:
#########################################################################
#### find enriched GOBP for consistent signals in inhibitory neurons ####
#########################################################################
res$gene_id <- mapIds(
  # Replace with annotation package for the organism relevant to your data
  org.Hs.eg.db,
  # The vector of gene identifiers we want to map
  keys = res$genes,
  # Replace with the type of gene identifiers in your data
  keytype = "SYMBOL",
  # Replace with the type of gene identifiers you would like to map to
  column = "ENTREZID",
  multiVals = "first"
)

In [ ]:
### Run enrichment analysis for overlapped upregulated genes in C1, C2, and C3 clusters
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  group_by(cluster)
sig_genes_list = split(sig_genes_list$gene_id, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
length(sig_genes_list)

# # ## get the common upregulated genes in C1, C2, and C3 clusters
# # common_genes = Reduce(intersect, sig_genes_list[c("C2",'C3','C1')])
# # length(unique(common_genes))

In [ ]:
# ## Run enrichment analysis for the common upregulated genes in C1, C2, and C3 clusters
# ego <- enrichGO(gene = common_genes,
#                 OrgDb = org.Hs.eg.db,
#                 keyType = "ENTREZID",
#                 ont = "BP",
#                 pAdjustMethod = "BH",
#                 pvalueCutoff = 0.05,
#                 qvalueCutoff = 0.05)
# ego_result <- as.data.frame(ego)
# ego_result <- ego_result %>%
#   arrange(p.adjust) %>%
#   filter(p.adjust < 0.05) %>%
#   slice_head(n = 10)
# ego_result

In [ ]:
# ## Run enrichment analysis for the unique upregulated genes in cluster of interest compared to all other clusters
# cluster_oi = c("C1","C2","C3")
# cluster_oi = c("C5")
# c_of_others = setdiff(names(sig_genes_list), cluster_oi)
# g_all_others = Reduce(union, sig_genes_list[c_of_others])
# unique_genes = setdiff(Reduce(intersect, sig_genes_list[cluster_oi]), g_all_others)
# length(unique_genes)

# # g_all_others = Reduce(union, sig_genes_list[c("C2")
# # g_all_others = Reduce(union, sig_genes_list[c("C2",'C3','C4','C5')])
# # unique_C1_genes = setdiff(sig_genes_list$C1, g_all_others)
# # length(unique_C1_genes)

# ## run enrichment analysis for the unique upregulated genes in C1 cluster
# ego <- enrichGO(gene = unique_genes,
#                 OrgDb = org.Hs.eg.db,
#                 keyType = "ENTREZID",
#                 ont = "BP",
#                 pAdjustMethod = "BH",
#                 readable      = TRUE,
#                 pvalueCutoff = 0.05,
#                 qvalueCutoff = 0.05)
# ego_result <- as.data.frame(ego@result)
# head(ego_result,10)

# ## Run reactome pathway analysis for the unique upregulated genes in cluster of interest compared to all other clusters
# era = enrichPathway(gene=unique_genes, pvalueCutoff = 0.05,readable = TRUE)
# era_result = as.data.frame(era@result)
# head(era_result,10)

In [ ]:
## run Reactome pathway analysis for each cluster and combine the results
# era_result = data.frame()

# for (i in unique(res$cluster)){
#     cluster_genes <- res %>%
#         dplyr::filter(cluster == i, logFC > 0.5, FDR < 0.05) %>%
#         pull(gene_id) %>%
#         unique()
#     print(length(cluster_genes))

#     era = enrichPathway(gene=cluster_genes, pvalueCutoff = 0.05,readable = TRUE)
#     if (is.null(era)){
#             print(paste("No significant results for",i))
#         }else{
#             ## remove redundancy
#             era_df = era@result
#             era_df$cluster = i
#         }
#     era_result <- rbind(era_result, era_df)
# }

ego_result = data.frame()

for (i in unique(res$cluster)){
    cluster_genes <- res %>%
        dplyr::filter(cluster == i, logFC > 0.5, FDR < 0.05) %>%
        pull(gene_id) %>%
        unique()
    print(length(cluster_genes))

    ego <- enrichGO(gene = cluster_genes,
                OrgDb = org.Hs.eg.db,
                keyType = "ENTREZID",
                ont = "BP",
                pAdjustMethod = "BH",
                readable      = TRUE,
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.05)
                
    if (is.null(ego)){
            print(paste("No significant results for",i))
        }else{
            ## remove redundancy
            ego_df = ego@result
            ego_df$cluster = i
        }
    ego_result <- rbind(ego_result, ego_df)
}

In [ ]:
write.csv(top_pathways, file = "./Results/Revision_2/DEG/Top_enriched_endothelial_cluster_GO.csv")

In [ ]:
# ##  Draw the heatmap top 5 significant pathways for each cluster and save the plot
# top_pathways <- era_result %>%
#   dplyr::filter(p.adjust < 0.05) %>%
#   group_by(cluster) %>%
#   arrange(p.adjust) %>%
#   slice_head(n = 5)
# top_pathways <- top_pathways %>%
#   ungroup() %>%
#   mutate(cluster = factor(cluster, levels = c("C1","C2","C3","C4","C5")))

# options(repr.plot.width = 8,repr.plot.height = 6)
# p1 <- ggplot(top_pathways, aes(x = cluster, y = Description, fill = -log10(p.adjust))) +
#   geom_tile() +
#   scale_fill_viridis(option = "viridis") +
#   theme_classic() +
#   theme(axis.text.x = element_text(angle = 45, hjust = 1))
# pdf(file = "./Results/Revision_2/DEG/Endothelial_rg_clusters_top_pathways_reactome_heatmap_WC5.pdf",width = 7.5,height = 6)
# p1
# dev.off()

##  Draw the heatmap top 5 significant pathways for each cluster and save the plot
top_pathways <- ego_result %>%
  dplyr::filter(p.adjust < 0.05) %>%
  group_by(cluster) %>%
  arrange(p.adjust) %>%
  slice_head(n = 5)
top_pathways <- top_pathways %>%
  ungroup() %>%
  mutate(cluster = factor(cluster, levels = c("C1","C2","C3","C4","C5")))

options(repr.plot.width = 6,repr.plot.height = 6)
p2 <- ggplot(top_pathways, aes(x = cluster, y = Description, fill = -log10(p.adjust))) +
  geom_tile() +
  scale_fill_viridis(begin = 0,end = 1,option = "viridis") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
# pdf(file = "./Results/Revision_2/DEG/Endothelial_rg_clusters_top_pathways_go_heatmap_WC5.pdf",width = 6,height = 6)
p2
# dev.off()

In [ ]:
table(y$samples$group,y$samples$donor)

### Enrichment analysis of marker genes

In [ ]:
suppressPackageStartupMessages({
library(clusterProfiler)
library('org.Hs.eg.db')
})

In [ ]:
res = wilcox_pairwise_sig
res$genes = res$gene
head(res)

## ------------------------------------------ ##
## Draw heatmap on clusters on selected genes ##
## ------------------------------------------ ##

In [ ]:
### Making sure get the corrected layer and matrix
DefaultAssay(obj_oi) = "RNA"
# obj_oi = JoinLayers(obj_oi)

### Aggregation to pseudobulk 
mtx = AggregateExpression(
        obj_oi, 
        # features = gene_oi,
        group.by = c("region_cluster","individualID"),
        assays = 'RNA',
        slot = "counts"
    ) 
mtx <- as.matrix(mtx$RNA)

In [ ]:
## Get the library size for each sample
lib_size <- Matrix::colSums(mtx)

In [ ]:
### Get the logCPM from the matrix
cpm <- t(t(mtx) / lib_size) * 1e6
logCPM <- log2(cpm + 1)

## Only focus on the genes of interest
gene_oi = c("ABCC1","ABCB1","ABCG2","SLC16A1","SLC2A1","SLC7A5","TFRC","INSR","IGF1R","CLDN5","OCLN","TJP1", "CAV1",'SLC7A1',"SLC22A5")

logCPM = logCPM[gene_oi,]
logCPM_z <- t(scale(t(logCPM)))

In [ ]:
#### Plot the genes of interest
# Main heatmap
set.seed(04)
ht <- Heatmap(
  logCPM_z,
  cluster_rows = TRUE,
  cluster_columns = TRUE,
#   column_km = 32,
#   row_km = 4,
  show_column_names = TRUE,
  show_row_names = TRUE,
  show_heatmap_legend = TRUE,
  use_raster = TRUE#,
  # cell_fun = function(j, i, x, y, w, h, fill) {
  #   if (sig_mat[i, j] < 0.05) {
  #     grid.text("*", x, y, gp = gpar(fontsize = 20, col = "black"), vjust = "center")
  #   }
  # }
)

# options(repr.plot.width = 30, repr.plot.height = 6, repr.plot.res = 200)
# ht

In [ ]:
### Calculate the averaged logCPM
# extract region (everything before first "_")
region <- sub("_.*$", "", colnames(logCPM))

# sanity check
table(region)

logCPM_region <- sapply(
  split(seq_len(ncol(logCPM)), region),
  function(i) {
    rowMeans(logCPM[, i, drop = FALSE])
  }
)
logCPM_region_z <- t(scale(t(logCPM_region)))

In [ ]:
## Provide the color code
meta <- obj_oi@meta.data
df <- unique(meta[,c("region_cluster")])

df = as.data.frame(df)
print(df)

df$region_color <- NA
df[df$df == "C1",]$region_color = "#D55E00"
df[df$df == "C2",]$region_color = "#009E73"
df[df$df == "C3",]$region_color = "#7d03cf"
df[df$df == "C4",]$region_color = "#0072B2"
df[df$df == "C5",]$region_color = "#e43500"
## Assign region color
region_color <- df$region_color
names(region_color) <-df$df


In [ ]:
hl <- HeatmapAnnotation(
  region = colnames(logCPM_region_z),
#   avg_density = avg_vec,
  col = list(
    region = region_color
    # avg_density = colorRamp2(
    #   c(min(avg_vec, na.rm = TRUE), max(avg_vec, na.rm = TRUE)),
    #   c("lightblue", "firebrick")
    # )
  ),
  annotation_name_side = "right"
)

# Bottom annotation: rotated brain region labels
bn <- HeatmapAnnotation(
  text = anno_text(
    colnames(logCPM_region_z),
    rot = 90,
    offset = unit(1, "npc"),
    just = "right"
  ),
  annotation_height = max_text_width(colnames(logCPM_region_z))
)

In [ ]:
# Main heatmap
ht <- Heatmap(
  logCPM_region_z,
  cluster_rows = TRUE,
  cluster_columns = TRUE,
  # column_km = 4,
  # row_km = 4,
  top_annotation = hl,
  bottom_annotation = bn,
  show_column_names = FALSE,
  show_row_names = TRUE,
  show_heatmap_legend = TRUE,
  use_raster = TRUE#,
  # cell_fun = function(j, i, x, y, w, h, fill) {
  #   if (sig_mat[i, j] < 0.05) {
  #     grid.text("*", x, y, gp = gpar(fontsize = 20, col = "black"), vjust = "center")
  #   }
  # }
)

In [ ]:
# Set display options (works in Jupyter or R notebooks)
options(repr.plot.width = 6, repr.plot.height = 6, repr.plot.res = 200)
set.seed(04)
ht

## ------------------------------------------ ##
## Draw heatmap on cell type on ct markers genes ##
## ------------------------------------------ ##

In [ ]:
### Making sure get the corrected layer and matrix
DefaultAssay(obj_oi) = "RNA"
# obj_oi = JoinLayers(obj_oi)

### Aggregation to pseudobulk 
mtx = AggregateExpression(
        obj_oi, 
        # features = gene_oi,
        group.by = c("Cell_type","individualID"),
        assays = 'RNA',
        slot = "counts"
    ) 
mtx <- as.matrix(mtx$RNA)

In [ ]:
## Get the library size for each sample
lib_size <- Matrix::colSums(mtx)

In [ ]:
### Get the logCPM from the matrix
cpm <- t(t(mtx) / lib_size) * 1e6
logCPM <- log2(cpm + 1)

## Only focus on the genes of interest
gene_oi = c("DKK2","VEGFC","ARL15","VCAM1","SLC7A5","INSR","RAMP3","TSHZ2","IL1R1","PLVAP","SLCO2A1","MKI67","CLDN5","FLT1")

logCPM = logCPM[gene_oi,]
logCPM_z <- t(scale(t(logCPM)))

In [ ]:
### Calculate the averaged logCPM
# extract region (everything before first "_")
region <- sub("_.*$", "", colnames(logCPM))

# sanity check
table(region)

logCPM_region <- sapply(
  split(seq_len(ncol(logCPM)), region),
  function(i) {
    rowMeans(logCPM[, i, drop = FALSE])
  }
)
logCPM_region_z <- t(scale(t(logCPM_region)))

In [ ]:
colnames(logCPM_region_z) <- gsub("-","_",colnames(logCPM_region_z))
logCPM_region_z = logCPM_region_z[,c("Large_Artery","Arterial","Capillary","Venous","Fenestrated_Capillary","EndoMT")]

In [ ]:
## Provide the color code
meta <- obj_oi@meta.data
df <- unique(meta[,c("Cell_type")])

df = as.data.frame(df)
print(df)

df$region_color <- NA
df[df$df == "Arterial",]$region_color = "#F4A582"
df[df$df == "Capillary",]$region_color = "#66C2A5"
df[df$df == "Venous",]$region_color = "#4393C3"
df[df$df == "Fenestrated_Capillary",]$region_color = "#FDAE61"
df[df$df == "EndoMT",]$region_color = "#BC80BD"
df[df$df == "Large_Artery",]$region_color = "#B2182B"
## Assign region color
region_color <- df$region_color
names(region_color) <-df$df


In [ ]:
hl <- HeatmapAnnotation(
  region = colnames(logCPM_region_z),
#   avg_density = avg_vec,
  col = list(
    region = region_color
    # avg_density = colorRamp2(
    #   c(min(avg_vec, na.rm = TRUE), max(avg_vec, na.rm = TRUE)),
    #   c("lightblue", "firebrick")
    # )
  ),
  annotation_name_side = "right"
)

# Bottom annotation: rotated brain region labels
bn <- HeatmapAnnotation(
  text = anno_text(
    colnames(logCPM_region_z),
    rot = 90,
    offset = unit(1, "npc"),
    just = "right"
  ),
  annotation_height = max_text_width(colnames(logCPM_region_z))
)

In [ ]:
# Main heatmap
ht <- Heatmap(
  logCPM_region_z,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  # column_km = 4,
  # row_km = 4,
  top_annotation = hl,
  bottom_annotation = bn,
  show_column_names = FALSE,
  show_row_names = TRUE,
  show_heatmap_legend = TRUE,
  use_raster = FALSE,
  col = colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))#,
  # cell_fun = function(j, i, x, y, w, h, fill) {
  #   if (sig_mat[i, j] < 0.05) {
  #     grid.text("*", x, y, gp = gpar(fontsize = 20, col = "black"), vjust = "center")
  #   }
  # }
)

In [ ]:
# Set display options (works in Jupyter or R notebooks)
options(repr.plot.width = 6, repr.plot.height = 6, repr.plot.res = 200)
set.seed(04)
pdf(file = "./Results/Revision_2/Figures/Figure_2b_Endothelial_cluster_DEG_examples_heatmap.pdf",width = 5.3,height = 6)
ht
dev.off()

## Porportional plot of region clusters across brain regions

In [ ]:
## get meta data to work on
meta <- obj_oi@meta.data

## get the count table
counts <- meta %>%
  group_by(Cell_type, region_cluster) %>%
  summarise(freq = n(), .groups = "drop")

colnames(counts) <- c("cell_type","region_cluster","freq")

## reorder the brain region based on the number code
counts <- counts[order(counts$cell_type),]
counts$region_cluster<- factor(counts$region_cluster, levels=unique(counts$region_cluster))

col1=c(
  "Large_Artery"     = "#B2182B",  # deep red
#   "Arterial"           = "#D6604D",  # red-orange
  "Arterial"        = "#F4A582",  # light orange
#   "Capillary1"       = "#A6DBD0",  # light teal
  "Capillary"       = "#66C2A5",  # medium teal
#   "Capillary"       = "#41AE76",  # deeper teal
  "Venous"           = "#4393C3",  # medium blue
#   "Venous"             = "#2166AC",  # dark blue
  "Fenestrated_Capillary"   = "#FDAE61",  # golden orange (distinct capillary subtype)
  "EndoMT"          = "#BC80BD"  # soft lavender purple
#   "EndoMT"          = "#762A83"   # deep plum purple
)

counts$color <- col1[counts$cell_type]

In [ ]:
p4 <- ggplot(counts, aes(fill=cell_type, y=freq, x=region_cluster)) + 
        geom_bar(position="fill", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 15),
              axis.text.y = element_text(size = 15),
              axis.title.y = element_text(size = 15)) +
        scale_fill_manual(values=col1) +
        xlab("") +
        ylab("Frequency")
options(repr.plot.width = 12, repr.plot.height = 6, repr.plot.res = 100) # set the resolution
# p3
p4

ggsave(filename = "./Results/Revision_2/Figures/Figure_1X_Endothelial_Cell_type_region_cluster_proportion.pdf", patchwork::wrap_plots(p4, ncol = 1),scale = 1, width = 8, height = 5)

## Cell type composition across brain regions

In [ ]:
## get meta data to work on
meta <- obj_oi@meta.data

## get the count table
counts <- meta %>%
  group_by(Cell_type, Region_combined) %>%
  summarise(freq = n(), .groups = "drop")
colnames(counts) <- c("Cell_type","brain_region","freq")

## reorder the brain region based on the number code
counts$order <- as.numeric(str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,1])
counts <- counts[order(counts$Cell_type),]
counts <- counts[order(counts$order),]

## making sure corrected order
counts$brain_region_plot = str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,2]
counts$brain_region_plot<- factor(counts$brain_region_plot, levels=unique(counts$brain_region_plot))


counts$Cell_type <- factor(counts$Cell_type)

col1=c(
  "Large_Artery"     = "#B2182B",  # deep red
#   "Arterial"           = "#D6604D",  # red-orange
  "Arterial"        = "#F4A582",  # light orange
#   "Capillary1"       = "#A6DBD0",  # light teal
  "Capillary"       = "#66C2A5",  # medium teal
#   "Capillary"       = "#41AE76",  # deeper teal
  "Venous"           = "#4393C3",  # medium blue
#   "Venous"             = "#2166AC",  # dark blue
  "Fenestrated_Capillary"   = "#FDAE61",  # golden orange (distinct capillary subtype)
  "EndoMT"          = "#BC80BD"  # soft lavender purple
#   "EndoMT"          = "#762A83"   # deep plum purple
)

counts$color <- col1[counts$Cell_type]

In [ ]:
p4 <- ggplot(counts, aes(fill=Cell_type, y=freq, x=brain_region_plot)) + 
        geom_bar(position="fill", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 15),
              axis.text.y = element_text(size = 15),
              axis.title.y = element_text(size = 15)) +
        scale_fill_manual(values=col1) +
        xlab("") +
        ylab("Frequency")
options(repr.plot.width = 12, repr.plot.height = 6.5, repr.plot.res = 100) # set the resolution
# p3
p4
# ggsave(filename = "./Results/Revision_2/Figures/Figure_1X_Endothelial_Cell_type_region_proportion.pdf", 
# patchwork::wrap_plots(p4, ncol = 1),
# scale = 1, width = 12, height = 7.5)

## Get psudo-bulk expression with AV bins

In [ ]:
### bin the pseudotime into 5 bins
N_AV_BINS = 5
qs <- quantile(obj_oi$pseudotime, probs = seq(0, 1, length.out = N_AV_BINS + 1))
labs <- paste0("AV", seq_len(N_AV_BINS))
obj_oi$AV_bin <- cut(obj_oi$pseudotime, breaks = qs, include.lowest = TRUE, labels = labs)

table(obj_oi$AV_bin,obj_oi$region_cluster)

In [ ]:
VlnPlot(obj_oi, features = c("pseudotime"), group.by = "AV_bin", pt.size = 0.5, combine = FALSE)

In [ ]:
## pseudobulk object generation for the AV bins and 
## getting the pseudo-bulk object
pseudo_obj <- AggregateExpression(obj_oi, assays = "RNA", return.seurat = T, group.by = c("sampleID","region_cluster","AV_bin"))
pseudo_obj

## getting the count matrix for edgeR analysis
mat = pseudo_obj[['RNA']]$counts
head(mat)

## getting the meta data for edgeR analysis
meta = pseudo_obj@meta.data
group = meta$region_cluster
head(meta)

In [ ]:
## Build edgeR format
y <- DGEList(counts=mat,group=group)
cpm_mat <- cpm(y, log=F)

## add donor info
df = str_split_fixed(rownames(y$samples),"-",5)[,c(1,2)]
y$samples$donor = paste(df[,1],df[,2],sep = "_")
df = str_split_fixed(rownames(y$samples),"_",3)
y$samples$AV_bin = df[,3]

## filter sample by number of reads
summary(y$samples$lib.size)

## Removing the samples with low number of reads lower than 5% quantile
cutoff <- quantile(y$samples$lib.size, 0.05)
print(cutoff)
keep.samples <- y$samples$lib.size > cutoff 
table(keep.samples)
y <- y[, keep.samples]
cpm_mat = cpm_mat[, keep.samples]

## preprocessing for model fitting
group <- droplevels(factor(y$samples$group))
donor <- droplevels(factor(y$samples$donor))
AV_bin <- droplevels(factor(y$samples$AV_bin))

design <- model.matrix(~0 + group + donor + AV_bin)
colnames(design) <- make.names(colnames(design))
head(design)

## filter genes based on design
keep <- filterByExpr(y, design=design)
table(keep)
y <- y[keep, , keep.lib.sizes=FALSE]

In [ ]:
## normalize library sizes
y <- normLibSizes(y)
summary(y$samples$norm.factors)

options(repr.plot.width=12,repr.plot.height=12)
brain_region <- as.factor(y$samples$group)
plotMDS(y, pch=16, col=c(1:36)[y$samples$group], main="brain region",labels = y$samples$donor)

In [ ]:
## estimate dispersion
y <- estimateDisp(y, design, robust=TRUE)
##--------- fit the model -----------##
fit <- glmQLFit(y, design, robust=TRUE)

In [ ]:
## define contrasts for each brain region vs all others
K <- nlevels(group)
grp_cols <- grep("^group", colnames(design), value=TRUE)

# sanity: length(grp_cols) should be K
stopifnot(length(grp_cols) == K)

C <- matrix(-1/(K-1), nrow=K, ncol=K,
            dimnames=list(grp_cols, levels(group)))
diag(C) <- 1

# expand to full design rows (add zeros for donor columns)
Cfull <- matrix(0, nrow=ncol(design), ncol=K,
                dimnames=list(colnames(design), colnames(C)))
Cfull[grp_cols, ] <- C

qlf <- vector("list", K)
for(i in seq_len(K)) {
  qlf[[i]] <- glmQLFTest(fit, contrast=Cfull[, i])
  qlf[[i]]$comparison <- paste0(levels(group)[i], "_vs_others")
}


In [ ]:
res = data.frame()
for (i in 1:length(qlf)){
    df = qlf[[i]]$table
    df$genes = rownames(df)
    df$FDR = p.adjust(df$PValue, method = "bonferroni")
    df %>%
        dplyr::arrange(FDR) %>%
        # dplyr::filter(FDR  < 0.05) %>%
        # dplyr::filter(logFC > 0) %>% 
        ungroup() -> sig

    if (nrow(sig) == 0) next    
    
    sig$cluster = levels(group)[i]
    res = rbind(res,sig)
}

In [ ]:
res$direction = ifelse(res$logFC > 0, "up", "down")
table(res$cluster,res$direction)

In [ ]:
## write the significant results
write.csv(res,file = "./Results/Revision_3/DEG/Endo_rg_clusters_AV_edgeR_wC5.csv")

In [ ]:
dt <- lapply(lapply(qlf, decideTests), summary)
dt.all <- do.call("cbind", dt)
dt.all

In [ ]:
res_sig = res %>% dplyr::filter(abs(logFC) > 0.25) %>% dplyr::filter(FDR < 0.05)
table(res_sig$cluster,res_sig$direction)

In [ ]:
write.csv(res_sig,file = "./Results/Revision_3/DEG/Endo_rg_clusters_AV_edgeR_wC5_sig.csv")

### Check the overlap of the significant genes across clusters

In [ ]:
## reloading the DEG results for the heatmap plot
res = read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_AV_edgeR_wC5.csv",row.names = 1)
head(res)

In [ ]:
## get the significant genes
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  group_by(cluster) 

## check number of significant genes in each cluster
table(sig_genes_list$cluster)

In [ ]:
## Draw upset plot for the significant genes across clusters
library(UpSetR)
options(repr.plot.height = 6,repr.plot.width = 12)

## prepare the data for upset plot
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  # dplyr::filter(logFC < -0.5) %>%
  group_by(cluster) 
sig_genes_list = split(sig_genes_list$genes, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
# length(sig_genes_list)

## draw the upset plot and add title "Number of upregulated DEGs in each region cluster"
pdf(file = "./Results/Revision_3/DEG/Endothelial_rg_clusters_AV_sig_genes_upset_plot_WC5.pdf",width = 12,height = 6)
upset(
  fromList(sig_genes_list), 
  order.by = "freq", 
  nsets = length(sig_genes_list), 
  nintersects =NA,
  main.bar.color = c("#FF0000","#0f0f0f","#CC79A7","#03B8D8","#009E73", "#009E73","#0f0f0f","#0f0f0f","#0f0f0f","#0f0f0f", "#0f0f0f","#0f0f0f","#0f0f0f"),
  sets.bar.color = c("#FF0000","#03B8D8", "#009E73", "#009E73", "#CC79A7"),
  text.scale = c(2, 2, 2, 2, 3, 2),
  point.size = 5, line.size = 1.6, mb.ratio = c(0.6, 0.4),
  att.pos = "bottom",
  mainbar.y.label = "Number of upregulated DEGs",
  sets.x.label = "Region Clusters"
  )
dev.off()

In [ ]:
sig_genes_list

## Draw Heatmap of top DE genes

In [ ]:
## optional: reloading the res
res = read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_AV_edgeR_wC5.csv", row.names = 1)
table(res$cluster,res$direction)

In [ ]:
## Get the top 10 significant genes for each cluster
res %>%
  filter(logFC > 0.5) %>%
  filter(FDR < 0.05) %>%
  group_by(cluster) %>%
  # arrange(desc(logFC)) %>%
  arrange(FDR) %>%
  slice_head(n = 10) -> top

top

# write.csv(top,file = "./Results/Revision_2/DEG/Endo_rg_clusters_edgeR_top20_wC5.csv")

In [ ]:
## working on the top regulated first
df_all = res

## construct the matrix
mat<-matrix(nrow=length(unique(top$genes)), ncol=length(unique(df_all$cluster)))
colnames(mat) <- unique(df_all$cluster)

### create the matrix to show the gene expression change pattern
for (j in 1:length(unique(top$genes))){
    for (i in 1:length(unique(df_all$cluster))){
        geneTemp<-unique(top$genes)[j]
        rgTemp<-colnames(mat)[i]

        sub<-df_all[which(df_all$genes==geneTemp & df_all$cluster==rgTemp),]
        if (nrow(sub)>0){
        mat[j,i]<-sub$logFC
        }
        else{
        mat[j,i]<-0
        }
    }  
}
rownames(mat)<- unique(top$genes)

#generating a significance matrix
sig_mat<-matrix(nrow=length(unique(top$genes)), ncol=length(unique(df_all$cluster)))
colnames(sig_mat)<-unique(df_all$cluster)

for (j in 1:length(unique(top$genes))){
    for (i in 1:length(unique(df_all$cluster))){
        geneTemp<-unique(top$genes)[j]
        rgTemp<-colnames(sig_mat)[i]
        sub<-df_all[which(df_all$genes==geneTemp & df_all$cluster==rgTemp),]
        if (nrow(sub)>0){
            sig_mat[j,i]<-sub$FDR
        }else{
        sig_mat[j,i]<-1
        }
    }
}
rownames(sig_mat)<- unique(top$genes)

## reorder the matrix
mat <- mat[,order(as.numeric(str_split_fixed(colnames(mat),pattern = "_",2)[,1]))]
sig_mat <- sig_mat[,order(as.numeric(str_split_fixed(colnames(sig_mat),pattern = "_",2)[,1]))]
# mat
# sig_mat

In [ ]:
# pdf("./Results/Revision_2/DEG/Figure_2X_Endothelial_DEG_heatmap_wC5_1.pdf",width = 4,height = 12)
ht=Heatmap(mat, cluster_rows=F,cluster_columns=F,
           col=colorRamp2(c(-3,0, 3),c("blue4","grey95","red4")),  
#           top_annotation=hl, #name="Number of genes with significant expression changes",  
          #  right_annotation = hl,
           show_column_names=T, show_row_names=T, column_title=NULL,
           row_names_side="left", row_title_gp=gpar(fontsize=60),
           row_names_max_width = max_text_width(rownames(mat), gp = gpar(fontsize = 24)),
           cell_fun = function(j, i, x, y, w, h, fill) {
  if(sig_mat[i, j] <0.05) {grid.text("*", x, y, gp=gpar(fontsize=35, col="black"), vjust="center")}
  #if(sig_mat[i, j] <0.05 & sig_mat[i, j] >0.01){grid.text("*", x, y, gp=gpar(fontsize=35, col="black"), vjust="center")}
})

ht
# dev.off()

In [ ]:
ht

In [ ]:
## Check if the significant genes within each cluster each direaction are the same or different
sig_og = read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_edgeR_wC5_sig.csv", row.names = 1)
sig_av = read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_AV_edgeR_wC5_sig.csv", row.names = 1)

table(sig_og$cluster,sig_og$direction)
table(sig_av$cluster,sig_av$direction)

## check the overlap between the significant genes in each cluster for the two different analyses


In [ ]:
## check the overlap between the significant genes in each cluster for the two different analyses
## ---------------------------------------------------------------------------
 
## gene IDs may live in a column or in the rownames depending on how the file
## was written -- resolve it once rather than assuming
get_gene <- function(df, nm) {
  for (cand in c("gene", "gene_name", "symbol", "genes", "SYMBOL")) {
    if (cand %in% colnames(df)) return(as.character(df[[cand]]))
  }
  if (!is.null(rownames(df)) && !all(grepl("^[0-9]+$", rownames(df)))) {
    return(as.character(rownames(df)))
  }
  stop(sprintf("Cannot find a gene column in %s. Columns: %s",
               nm, paste(colnames(df), collapse = ", ")))
}
sig_og$gene <- get_gene(sig_og, "sig_og")
sig_av$gene <- get_gene(sig_av, "sig_av")
 
## a gene should appear once per cluster; duplicates would silently inflate overlaps
stopifnot(!any(duplicated(paste(sig_og$gene, sig_og$cluster))),
          !any(duplicated(paste(sig_av$gene, sig_av$cluster))))
 
clusters   <- sort(union(unique(sig_og$cluster), unique(sig_av$cluster)))
directions <- c("up", "down")
 
## ---- 1. overlap per cluster x direction -----------------------------------
ov <- do.call(rbind, lapply(clusters, function(cl) {
  do.call(rbind, lapply(directions, function(dr) {
    g_og <- sig_og$gene[sig_og$cluster == cl & sig_og$direction == dr]
    g_av <- sig_av$gene[sig_av$cluster == cl & sig_av$direction == dr]
    shared <- intersect(g_og, g_av)
    uni    <- union(g_og, g_av)
    data.frame(
      cluster    = cl,
      direction  = dr,
      n_og       = length(g_og),
      n_av       = length(g_av),
      n_shared   = length(shared),
      n_og_only  = length(setdiff(g_og, g_av)),
      n_av_only  = length(setdiff(g_av, g_og)),
      jaccard    = if (length(uni)) length(shared) / length(uni) else NA_real_,
      pct_og_kept = if (length(g_og)) 100 * length(shared) / length(g_og) else NA_real_,
      stringsAsFactors = FALSE
    )
  }))
}))
ov$jaccard     <- round(ov$jaccard, 3)
ov$pct_og_kept <- round(ov$pct_og_kept, 1)
cat("\n== Overlap per cluster x direction ==\n"); print(ov, row.names = FALSE)

In [ ]:
## ---- 2. overlap per cluster, ignoring direction ---------------------------
ov_cl <- do.call(rbind, lapply(clusters, function(cl) {
  g_og <- sig_og$gene[sig_og$cluster == cl]
  g_av <- sig_av$gene[sig_av$cluster == cl]
  shared <- intersect(g_og, g_av); uni <- union(g_og, g_av)
  data.frame(cluster = cl, n_og = length(g_og), n_av = length(g_av),
             n_shared = length(shared),
             jaccard = round(if (length(uni)) length(shared)/length(uni) else NA, 3),
             pct_og_kept = round(if (length(g_og)) 100*length(shared)/length(g_og) else NA, 1),
             stringsAsFactors = FALSE)
}))
cat("\n== Overlap per cluster (direction ignored) ==\n"); print(ov_cl, row.names = FALSE)
 
## ---- 3. sign flips: significant in BOTH but opposite direction ------------
## these matter more than counts -- a flip means the AV adjustment reversed the
## conclusion, not merely weakened it
m <- merge(sig_og[, c("gene", "cluster", "direction", "logFC")],
           sig_av[, c("gene", "cluster", "direction", "logFC")],
           by = c("gene", "cluster"), suffixes = c("_og", "_av"))
flips <- m[m$direction_og != m$direction_av, ]
cat("\n== Sign flips (sig in both, opposite direction) ==\n")
cat("total:", nrow(flips), "of", nrow(m), "genes significant in both\n")
if (nrow(flips)) print(table(flips$cluster))
 
## ---- 4. concordance of effect sizes among shared genes --------------------
cat("\n== logFC concordance among genes significant in both ==\n")
conc <- do.call(rbind, lapply(clusters, function(cl) {
  s <- m[m$cluster == cl, ]
  data.frame(cluster = cl, n = nrow(s),
             pearson  = if (nrow(s) > 2) round(cor(s$logFC_og, s$logFC_av), 3) else NA,
             spearman = if (nrow(s) > 2) round(cor(s$logFC_og, s$logFC_av,
                                                   method = "spearman"), 3) else NA,
             median_abs_shrink = if (nrow(s)) round(median(abs(s$logFC_av)) -
                                                    median(abs(s$logFC_og)), 3) else NA,
             stringsAsFactors = FALSE)
}))
print(conc, row.names = FALSE)

In [ ]:
met.brewer("Demuth",3)

In [ ]:
## =============================================================================
## Compare significant genes between the original and AV-adjusted DEG analyses.
## Two heatmaps: one for upregulated genes, one for downregulated.
##
## Rows    = genes significant in that direction in EITHER analysis
## Columns = cluster, paired as OG | AV side by side
## Fill    = logFC where significant, grey where not significant in that analysis
##
## Reading the figure: a row that is coloured in the OG column and grey in the
## AV column next to it is a gene whose regional effect did not survive AV
## adjustment. Grey-then-coloured is a gene the naive analysis missed.
## =============================================================================

library(pheatmap)
library(RColorBrewer)
library(MetBrewer)

sig_og <- read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_edgeR_wC5_sig.csv", row.names = 1)
sig_av <- read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_AV_edgeR_wC5_sig.csv", row.names = 1)

outdir <- "./Results/Revision_3/DEG"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

## ---- gene IDs may be a column or the rownames -------------------------------
get_gene <- function(df, nm) {
  for (cand in c("gene", "gene_name", "symbol", "genes", "SYMBOL"))
    if (cand %in% colnames(df)) return(as.character(df[[cand]]))
  if (!is.null(rownames(df)) && !all(grepl("^[0-9]+$", rownames(df))))
    return(as.character(rownames(df)))
  stop(sprintf("Cannot find a gene column in %s. Columns: %s", nm,
               paste(colnames(df), collapse = ", ")))
}
sig_og$gene <- get_gene(sig_og, "sig_og")
sig_av$gene <- get_gene(sig_av, "sig_av")
stopifnot(!any(duplicated(paste(sig_og$gene, sig_og$cluster))),
          !any(duplicated(paste(sig_av$gene, sig_av$cluster))))

clusters <- sort(union(unique(sig_og$cluster), unique(sig_av$cluster)))

## =============================================================================
## build one matrix per direction
## =============================================================================
build_mat <- function(dir_keep) {

  ## rows are chosen by direction, but values are filled from the FULL result
  ## tables regardless of direction. That way a gene up in OG and down in AV
  ## shows red | blue -- a visible sign flip -- instead of red | grey, which
  ## would be indistinguishable from simply losing significance.
  genes <- sort(union(sig_og$gene[sig_og$direction == dir_keep],
                      sig_av$gene[sig_av$direction == dir_keep]))
  if (!length(genes)) return(NULL)
  og <- sig_og
  av <- sig_av

  ## columns interleaved OG, AV per cluster so the pair sits side by side
  cols <- as.vector(rbind(paste0(clusters, "_OG"), paste0(clusters, "_AV")))
  m <- matrix(NA_real_, nrow = length(genes), ncol = length(cols),
              dimnames = list(genes, cols))

  fill <- function(m, d, suffix) {
    d <- d[d$gene %in% genes, ]
    if (!nrow(d)) return(m)
    idx <- cbind(match(d$gene, genes), match(paste0(d$cluster, suffix), cols))
    m[idx] <- d$logFC
    m
  }
  m <- fill(m, og, "_OG")
  m <- fill(m, av, "_AV")

  ## drop genes significant in only one cluster-analysis cell -- they carry no
  ## comparison and just add height. Set min_cells = 1 to keep everything.
  min_cells <- 2
  keep <- rowSums(!is.na(m)) >= min_cells
  if (sum(keep) < 10) keep <- rep(TRUE, nrow(m))   # don't empty the matrix
  m <- m[keep, , drop = FALSE]

  ## order rows so shared genes group together: sort by the pattern of which
  ## cells are significant, then by mean logFC
  pat <- apply(!is.na(m), 1, function(r) paste(as.integer(r), collapse = ""))
  m <- m[order(pat, -rowMeans(m, na.rm = TRUE)), , drop = FALSE]
  m
}

## =============================================================================
## ALTERNATIVE VIEW: cluster x cluster Jaccard between the two analyses.
## Use this if what you want is "does cluster identity survive adjustment"
## rather than gene-level detail. Rows = OG clusters, cols = AV clusters;
## a strong diagonal means each cluster's signature maps to itself.
## =============================================================================
jaccard_hm <- function(dir_keep, file) {
  og <- sig_og[sig_og$direction == dir_keep, ]
  av <- sig_av[sig_av$direction == dir_keep, ]
  J <- outer(clusters, clusters, Vectorize(function(a, b) {
    ga <- og$gene[og$cluster == a]; gb <- av$gene[av$cluster == b]
    u <- length(union(ga, gb)); if (u == 0) NA else length(intersect(ga, gb)) / u
  }))
  dimnames(J) <- list(paste0("OG_", clusters), paste0("AV_", clusters))
  pheatmap(J, cluster_rows = FALSE, cluster_cols = FALSE,
            ## change the color to met.brewer("Demuth", 3) or brewer.pal(9, "Purples")[1:6] if you want a purple scale
           color = colorRampPalette(brewer.pal(9, "Purples")[1:7])(100),
           display_numbers = matrix(sprintf("%.2f", J), nrow=nrow(J)), number_color = "grey15", fontsize_number = 10, legend = TRUE,
           main = sprintf("Jaccard overlap, %sregulated",
                          ifelse(dir_keep == "up", "up", "down")),
           filename = file, width = 5.2, height = 4.6, fontsize = 9)
  cat("wrote", file, "\n")
  J
}
for (dr in c("up", "down")) {
  jaccard_hm(dr, file.path(outdir, sprintf("Endo_rg_OG_vs_AV_jaccard_%s.pdf", dr)))
}

## Enrichment analysis for the significant genes in each cluster of AV analysis

In [ ]:
res = read.csv("./Results/Revision_3/DEG/Endo_rg_clusters_AV_edgeR_wC5.csv",row.names = 1)
res_sig = res %>% dplyr::filter(FDR < 0.05) %>% dplyr::filter(logFC > 0.5)
table(res_sig$cluster)
# res

In [ ]:
suppressPackageStartupMessages({
library(clusterProfiler)
library('org.Hs.eg.db')
# library(ReactomePA)
})

In [ ]:
#########################################################################
#### find enriched GOBP for consistent signals in inhibitory neurons ####
#########################################################################
res$gene_id <- mapIds(
  # Replace with annotation package for the organism relevant to your data
  org.Hs.eg.db,
  # The vector of gene identifiers we want to map
  keys = res$genes,
  # Replace with the type of gene identifiers in your data
  keytype = "SYMBOL",
  # Replace with the type of gene identifiers you would like to map to
  column = "ENTREZID",
  multiVals = "first"
)

In [ ]:
### Run enrichment analysis for overlapped upregulated genes in C1, C2, and C3 clusters
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  group_by(cluster)
sig_genes_list = split(sig_genes_list$gene_id, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
length(sig_genes_list)

In [ ]:
ego_result <- data.frame()
for (i in unique(res$cluster)) {
 
  cluster_genes <- res %>%
    dplyr::filter(cluster == i, logFC > 0.5, FDR < 0.05) %>%
    pull(gene_id) %>% unique()
  message(i, ": ", length(cluster_genes), " genes")
 
  if (length(cluster_genes) < 10) {           # too few to enrich meaningfully
    message("  skipping ", i, " -- too few genes"); next
  }
 
  ego <- enrichGO(gene = cluster_genes, OrgDb = org.Hs.eg.db,
                  keyType = "ENTREZID", ont = "BP", pAdjustMethod = "BH",
                  readable = TRUE, pvalueCutoff = 0.05, qvalueCutoff = 0.05)
 
  if (is.null(ego) || nrow(ego@result) == 0) {
    message("  no significant results for ", i); next            # <- the fix
  }
 
  ## GO BP is heavily redundant -- parent and child terms both come up and eat
  ## the top-5 slots with what is effectively one finding. This is what the
  ## "remove redundancy" comment intended.
  ego <- clusterProfiler::simplify(ego, cutoff = 0.5, by = "p.adjust",
                                   select_fun = min)
 
  ego_df <- ego@result
  ego_df$cluster <- i
  ego_result <- rbind(ego_result, ego_df)
}

In [ ]:
library(tidyr)
library(forcats)

TOP_N     <- 5
CLUSTER_LEVELS <- c("C1","C2","C3","C4","C5")
outdir    <- "./Results/Revision_3/DEG"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)
 
ego_result <- ego_result %>%
  mutate(cluster = factor(cluster, levels = CLUSTER_LEVELS))
 
## which terms to show
top_terms <- ego_result %>%
  dplyr::filter(p.adjust < 0.05) %>%
  group_by(cluster) %>%
  arrange(p.adjust, .by_group = TRUE) %>%     # .by_group makes the intent explicit
  slice_head(n = TOP_N) %>%
  ungroup()
 
## order terms by the cluster they are top in, then by significance, so the
## figure reads roughly block-diagonal instead of alphabetically scrambled
term_order <- top_terms %>%
  group_by(Description) %>%
  slice_min(p.adjust, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  arrange(cluster, p.adjust) %>%
  pull(Description) %>% unique()
 
## complete matrix: every selected term looked up in every cluster
plot_df <- ego_result %>%
  dplyr::filter(Description %in% term_order) %>%
  mutate(
    gene_ratio = sapply(strsplit(GeneRatio, "/"),
                        function(x) as.numeric(x[1]) / as.numeric(x[2])),
    neglog10q  = -log10(p.adjust)
  ) %>%
  dplyr::select(Description, cluster, gene_ratio, Count, p.adjust, neglog10q) %>%
  complete(Description, cluster) %>%                  # fills absent combos with NA
  mutate(
    Description = factor(Description, levels = rev(term_order)),
    sig         = !is.na(p.adjust) & p.adjust < 0.05,
    label_wrap  = str_wrap(as.character(Description), width = 42)
  )
plot_df$label_wrap <- factor(plot_df$label_wrap,
                             levels = str_wrap(rev(term_order), width = 42))

In [ ]:
# ## ---- A. dot plot: size = gene ratio, colour = significance ------------------
# ## Two encodings instead of one. Colour alone cannot distinguish a term driven
# ## by 4 genes from one driven by 40, and that difference matters.
# p_dot <- ggplot(plot_df %>% dplyr::filter(sig),
#                 aes(x = cluster, y = label_wrap)) +
#   geom_point(data = plot_df %>% dplyr::filter(!sig),
#              aes(x = cluster, y = label_wrap),
#              shape = 4, size = 1.1, colour = "grey82", inherit.aes = FALSE) +
#   geom_point(aes(size = gene_ratio, fill = neglog10q),
#              shape = 21, colour = "white", stroke = 0.3) +
#   scale_fill_gradient(low = "#FEE5D9", high = "#67000D",
#                     name = expression(-log[10]~italic(q))) +
#   scale_size_continuous(range = c(1.6, 6.5), name = "gene ratio",
#                         labels = scales::percent_format(accuracy = 1)) +
#   scale_x_discrete(drop = FALSE, position = "top") +
#   labs(x = NULL, y = NULL,
#        title = sprintf("Top %d GO:BP terms per cluster (upregulated genes)", TOP_N),
#        caption = paste("x = not returned as enriched at q < 0.05 in that cluster.",
#                        "\nenrichGO only returns terms passing its cutoffs, so this is not",
#                        "\nthe same as a measured null -- rerun with pvalueCutoff = 1 if you",
#                        "\nneed the actual q-value everywhere.")) +
#   theme_bw(base_size = 9) +
#   theme(panel.grid.major = element_line(colour = "grey93", linewidth = 0.3),
#         panel.grid.minor = element_blank(),
#         panel.border     = element_rect(colour = "grey80", linewidth = 0.4),
#         axis.text.y      = element_text(size = 7, lineheight = 0.85),
#         axis.text.x      = element_text(size = 8, face = "bold"),
#         axis.ticks       = element_blank(),
#         plot.title       = element_text(size = 9.5, hjust = 0),
#         plot.caption     = element_text(size = 6.5, colour = "grey40"),
#         legend.key.size  = unit(0.35, "cm"),
#         legend.title     = element_text(size = 7.5),
#         legend.text      = element_text(size = 6.5))
 
n_terms <- length(term_order)
# ggsave(file.path(outdir, "Endothelial_rg_clusters_AV_top_pathways_go_dotplot_WC5.pdf"),
#        p_dot, width = 7.2, height = max(3.5, 0.30 * n_terms + 1.6),
#        device = cairo_pdf)
 
## ---- B. tile version, kept for comparison -----------------------------------
## Same complete matrix, but grey for tested-not-significant so blank never
## silently means two different things.
p_tile <- ggplot(plot_df, aes(x = cluster, y = label_wrap, fill = neglog10q)) +
  geom_tile(colour = "white", linewidth = 0.6) +
  scale_fill_gradient(low = "#FEE5D9", high = "#a0061b",na.value = "transparent",
                    name = expression(-log[10]~italic(q))) +
#  scale_fill_viridis(begin = 0,end = 1,option = "viridis") +
  scale_x_discrete(drop = FALSE) +
  labs(x = NULL, y = NULL,
       title = sprintf("Top %d GO:BP terms per cluster", TOP_N)) +
  theme_classic() +
  theme(panel.grid    = element_blank(),
        axis.text.y   = element_text(size = 11, lineheight = 0.85),
        axis.text.x   = element_text(size = 11, face = "plain"),
        plot.caption  = element_text(size = 6.5, colour = "grey40"),
        legend.key.size = unit(0.35, "cm"))
 
ggsave(file.path(outdir, "Endothelial_rg_clusters_AV_top_pathways_go_heatmap_WC5.pdf"),
       p_tile, width = 7, height = 8,
       device = cairo_pdf)
 
cat("terms shown:", n_terms, "\n")
print(plot_df %>% group_by(cluster) %>%
        summarise(n_sig = sum(sig), .groups = "drop"))

In [ ]:
write.csv(ego_result, file = "./Results/Revision_3/DEG/Enriched_endothelial_cluster_AV_GO.csv")

## Get psudo-bulk expression

In [ ]:
sessionInfo()